# moe-engine — NVIDIA A100 Validation (Lightning AI Studio)

This notebook validates moe-engine's Triton router kernel, MoE layer, and
full test suite on a real **NVIDIA A100** GPU via **Lightning AI Studio**.

It is a sibling to `notebooks/moe_engine_v032_T4_validation.ipynb`, not a
replacement for it — that notebook stays in the repository unchanged and
validates the same codebase on a T4 (Turing). This notebook exists because
the T4 and A100 runs surfaced a real, documented difference between GPU
generations; see **ADR-005** (`docs/adr/ADR-005-gpu-architecture-portability.md`)
for the full writeup of what was found and fixed.

**This notebook does not target a fixed version number.** It validates
whatever code is currently checked out from the repository — Section 1
below clones the repo and Section 2 prints the actual installed package
version at runtime, so this notebook's claims never go stale the way a
hardcoded version string in markdown would.

**Platform notes specific to Lightning AI Studio** (read this before running):

- There is **no `/content/` directory** on Lightning AI Studio (that is a
  Google Colab convention). Every output path in this notebook is built
  from a portable `ARTIFACT_DIR` variable set once in Section 1 and reused
  everywhere — nothing is hardcoded to a platform-specific path.
- The working directory convention here is `/teamspace/studios/this_studio`,
  Lightning's default studio root. If you're running this somewhere else,
  Section 1 will still work correctly since it derives paths from
  `Path.cwd()` rather than assuming a fixed root.
- GPU: this notebook expects an A100. It will still run on other GPUs
  (the code has no A100-specific branches — see ADR-005 for why), but the
  benchmark numbers in the markdown below this cell are specifically A100
  numbers and won't match on different hardware.

**Before running:** select a GPU-enabled Studio (A100 recommended, matching
what this notebook was validated on).


## 1. Environment setup, portable artifact directory, and GPU check

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ---- Portable artifact directory (NOT /content -- that's Colab-only) ----
# Every output path in this notebook is built from ARTIFACT_DIR. This
# directory is created relative to the current working directory, so it
# works identically on Lightning AI Studio, Colab, a local machine, or any
# other environment -- no platform-specific assumptions.
ARTIFACT_DIR = Path.cwd() / "moe_engine_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be written to: {ARTIFACT_DIR}")
print(f"Current working directory:    {Path.cwd()}")


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print()
print('torch.__version__       =', torch.__version__)
print('torch.cuda.is_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('torch.cuda.device_name  =', torch.cuda.get_device_name(0))
    print('torch.version.cuda      =', torch.version.cuda)
    cap = torch.cuda.get_device_capability(0)
    print('compute capability      =', cap)
else:
    raise RuntimeError('No CUDA device found. Select a GPU-enabled Studio and restart.')


### Known-validated software stack (informational, non-blocking)

Per **ADR-005**, only one A100 software stack has been observed so far
(`torch==2.8.0+cu128`, Lightning AI Studio, July 2026). A hard version pin
derived from a single data point isn't well-founded yet, so the cell below
**prints and soft-warns only** — it will not stop the notebook if your
versions differ. If you hit an unexpected Triton compilation error later
in this notebook, the versions printed here are the first thing to check
against `docs/adr/ADR-005-gpu-architecture-portability.md`'s support
matrix, and worth adding to that table via a PR either way — this is
exactly how the matrix grows.


In [ ]:
_KNOWN_GOOD_TORCH_PREFIX = "2.8"  # from the July 2026 A100 validation run

if not torch.__version__.startswith(_KNOWN_GOOD_TORCH_PREFIX):
    print(f"NOTE: installed torch ({torch.__version__}) differs from the torch "
          f"version this notebook was last validated against ({_KNOWN_GOOD_TORCH_PREFIX}.x).")
    print("This is informational only -- proceeding. See ADR-005 if something breaks below.")
else:
    print(f"torch {torch.__version__} matches the last-validated A100 stack. Proceeding.")


## 2. Clone the repository and install

Unlike the T4/Colab workflow (which uploads a zip), this notebook clones
directly from GitHub -- set `REPO_URL` below to whichever remote you're
validating. Default is the main repository; swap in a fork's URL here if
you're testing changes before opening a PR (this is exactly how the A100
portability fixes in ADR-005 were originally developed and validated,
before being merged upstream).


In [ ]:
# Default: main repository. Override with a fork's URL to test changes
# pre-PR, e.g. a personal testing fork:
#   REPO_URL = "https://github.com/<your-username>/Composed-Mixture-of-Experts-Engine.git"
REPO_URL = "https://github.com/Mattral/Composed-Mixture-of-Experts-Engine.git"
REPO_DIR = Path.cwd() / "Composed-Mixture-of-Experts-Engine"

if REPO_DIR.exists():
    print(f"{REPO_DIR} already exists -- pulling latest instead of re-cloning.")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

PROJECT_ROOT = REPO_DIR / "moe-engine"
assert PROJECT_ROOT.exists(), f"Expected {PROJECT_ROOT} to exist after clone"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Working directory: {Path.cwd()}")


In [ ]:
!pip install -q -e ".[dev]"
!pip install -q triton pydantic typer
print("Installation complete.")


## 3. Confirm the ADR-005 fixes are present

This checks the code on disk for the specific fixes documented in
**ADR-005** before running anything: the `MIN_TRITON_DIM` dimension guard,
the single-source-of-truth `_should_use_triton` helper, and the
non-differentiable `report` side channel that keeps `used_triton`
telemetry honest. This replaces what used to be a notebook cell that
patched the source file at runtime with string replacement -- that was a
reasonable way to iterate live while developing the fix, but the fix now
lives permanently in `pkg/kernels/moe_router.py` in the repository, so no
patching is needed here. This cell verifies the fix is present; it does
not apply it.


In [ ]:
import pathlib

router_src = pathlib.Path('pkg/kernels/moe_router.py').read_text()

checks = [
    ('MIN_TRITON_DIM constant present',
     'MIN_TRITON_DIM = 16' in router_src),
    ('_should_use_triton single-source-of-truth helper present',
     'def _should_use_triton(' in router_src),
    ('report side-channel threaded through forward()',
     'report=None' in router_src and 'report["used_triton"]' in router_src),
    ('used_triton reads from report dict (not the old incomplete check)',
     'used_triton=_router_report.get("used_triton"' in router_src),
    ('K: tl.constexpr fix present (ADR-001)',
     router_src.count('K: tl.constexpr,') >= 2),
]

print('ADR-005 / ADR-001 fix verification:')
print('=' * 60)
passed = 0
for desc, ok in checks:
    status = '\033[92mPASS\033[0m' if ok else '\033[91mFAIL\033[0m'
    print(f'  [{status}] {desc}')
    if ok:
        passed += 1
print(f'\n{passed}/{len(checks)} checks passed.')
if passed < len(checks):
    raise RuntimeError(
        'Not all ADR-005 fixes are present in this checkout. '
        'Check REPO_URL in Section 2 points to a branch with these fixes merged.'
    )


## 4. Package version (read dynamically, not hardcoded)

This notebook does not assert a specific version number anywhere -- it
prints whatever is actually installed, from the single source of truth
(`pkg.__version__`), so this section never goes stale.


In [ ]:
import pkg
print(f"moe-engine version: {pkg.__version__}")
print(f"(validated at time of writing against this exact checkout on A100 -- "
      f"see ADR-005 for the full validation record)")


## 5. Smoke test — real Triton GPU path end-to-end

Exercises `MoEConfig` → `build_topology` → `ToyMoEModel` →
`DistributedMoELayer` → `MoERouter` (Triton kernel on CUDA) →
`ElasticTrainerHarness` checkpoint → `StructuredLogger`, in one pass.


In [ ]:
!python train.py --config configs/smoke.yaml --smoke


## 6. Config validation (all shipped configs, including large_scale.yaml)

In [ ]:
!python scripts/cli.py validate configs/


## 7. Full test suite on real GPU hardware

The 348-test Tier-0 CPU suite normally runs in CPU-only CI. Running it here,
in an environment with a real GPU actually present, is a meaningfully
different check: several tests branch on `torch.cuda.is_available()` and
exercise code paths that CPU-only CI cannot reach at all.


In [ ]:
import os
os.environ['ARTIFACT_DIR'] = str(ARTIFACT_DIR)

!pytest tests/ \
  --ignore=tests/test_chaos.py \
  --ignore=tests/test_smoke_e2e.py \
  -m cpu \
  -k "not (2rank or multiprocess or distributed_invariants)" \
  -q --tb=short 2>&1 | tee "$ARTIFACT_DIR/test_suite_output.txt" | tail -20


In [ ]:
!pytest tests/test_kernels.py tests/test_kernels_numerics.py -v --tb=short 2>&1 | tail -25


## 8. Direct demonstration of the ADR-005 fixes on real A100 hardware

Two things get proven here directly, not just inferred from "the test suite
passed": (1) a tiny-`E` configuration that would previously crash Triton
compilation on Ampere now runs cleanly via the reference-path fallback, and
(2) `used_triton` correctly reports `False` for it and `True` for a
normal-size configuration -- i.e., the telemetry is provably honest, not
just "probably fine."


In [ ]:
import torch
from pkg.kernels.moe_router import MoERouter, MIN_TRITON_DIM

print(f"MIN_TRITON_DIM = {MIN_TRITON_DIM}\n")

# --- Case 1: E=4, well under MIN_TRITON_DIM -- this exact shape is what
#     configs/smoke.yaml uses, and is what failed Triton compilation on
#     A100 before the ADR-005 fix. ---
router_tiny = MoERouter(hidden_dim=32, num_experts=4, top_k=2).cuda()
tokens_tiny = torch.randn(16, 32, device='cuda')
idx, w, cnt = router_tiny(tokens_tiny)
print(f"[E=4 config]  used_triton={router_tiny.last_profile.used_triton}  "
      f"(expected False -- below MIN_TRITON_DIM, must not crash, must not claim Triton ran)")
assert router_tiny.last_profile.used_triton is False
assert int(cnt.sum().item()) == 16 * 2, "token conservation must still hold on the reference path"
print("  -> No crash. Token conservation holds. used_triton correctly reports False.\n")

# --- Case 2: production-scale dims, comfortably above MIN_TRITON_DIM ---
router_full = MoERouter(hidden_dim=256, num_experts=32, top_k=2).cuda()
tokens_full = torch.randn(128, 256, device='cuda')
idx2, w2, cnt2 = router_full(tokens_full)
print(f"[E=32 config] used_triton={router_full.last_profile.used_triton}  "
      f"(expected True -- comfortably above MIN_TRITON_DIM on a real GPU)")
assert router_full.last_profile.used_triton is True
print("  -> Triton path ran as expected, and correctly reported so.\n")

print("Both cases match ADR-005's documented contract exactly.")


## 9. GPU benchmark sweep (A100)

In [ ]:
!python benchmarks/run_benchmark.py \
    --cuda \
    --json "$ARTIFACT_DIR/gpu_results_a100.json" \
    --csv  "$ARTIFACT_DIR/gpu_results_a100.csv"
print()
print("GPU benchmark complete.")


## 10. CPU benchmark sweep (for CPU vs A100 comparison)

In [ ]:
!python benchmarks/run_benchmark.py \
    --json "$ARTIFACT_DIR/cpu_results.json" \
    --csv  "$ARTIFACT_DIR/cpu_results.csv"
print()
print("CPU benchmark complete.")


## 11. CPU vs A100 throughput comparison

In [ ]:
import json

cpu_data = json.load(open(ARTIFACT_DIR / 'cpu_results.json'))
gpu_data = json.load(open(ARTIFACT_DIR / 'gpu_results_a100.json'))

cpu_map = {(d['name'], d['N'], d['H'], d['E'], d['K']): d
           for d in cpu_data if d['name'].startswith('router')}
gpu_map = {(d['name'], d['N'], d['H'], d['E'], d['K']): d
           for d in gpu_data if d['name'].startswith('router')}

print(f"{'Config':<38} {'CPU (M tok/s)':>14} {'A100 (M tok/s)':>15} {'Speedup':>10}")
print('-' * 82)

for key in sorted(set(cpu_map) & set(gpu_map)):
    name, n, h, e, k = key
    c = cpu_map[key]['tokens_per_sec']
    g = gpu_map[key]['tokens_per_sec']
    tag = f'{name} N={n},H={h},E={e},K={k}'
    speedup = g / c
    print(f'  {tag:<36} {c/1e6:>13.3f} {g/1e6:>14.3f} {speedup:>9.1f}x')


## 12. Throughput chart (A100)

Saved as `router_throughput_A100.png` -- deliberately a different filename
from the T4 notebook's `router_throughput_gpu_v0_3_2.png`, so generating
both doesn't silently overwrite one with the other if a user runs both
notebooks into the same artifact directory.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

router_fwd_gpu = [d for d in gpu_data if d['name'] == 'router_fwd' and d['device'] == 'cuda']
router_bwd_gpu = [d for d in gpu_data if d['name'] == 'router_fwd_bwd' and d['device'] == 'cuda']
router_fwd_cpu = [d for d in cpu_data if d['name'] == 'router_fwd' and d['device'] == 'cpu']

configs = [(d['N'], d['H'], d['E'], d['K']) for d in router_fwd_gpu]
labels = [f'N={n}\nH={h},E={e},K={k}' for n, h, e, k in configs]
x = list(range(len(configs)))

fwd_gpu = [d['tokens_per_sec'] / 1e6 for d in router_fwd_gpu]
bwd_gpu = [d['tokens_per_sec'] / 1e6 for d in router_bwd_gpu]
fwd_cpu = [d['tokens_per_sec'] / 1e6 for d in router_fwd_cpu]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, fwd_gpu, marker='o', linewidth=2.2, color='#1f77b4',
        label='router_fwd (Triton, A100)')
ax.plot(x, bwd_gpu, marker='s', linewidth=2.2, color='#ff7f0e',
        label='router_fwd_bwd (Triton, A100)')
ax.plot(x, fwd_cpu, marker='^', linewidth=1.5, color='#2ca02c',
        linestyle='--', label='router_fwd (fp64 ref, CPU)')

max_idx = int(np.argmax([g / c for g, c in zip(fwd_gpu, fwd_cpu)]))
speedup = fwd_gpu[max_idx] / fwd_cpu[max_idx]
ax.annotate(f'{speedup:.0f}x vs CPU',
            xy=(max_idx, fwd_gpu[max_idx]),
            xytext=(max_idx - 0.4, fwd_gpu[max_idx] * 1.5),
            fontsize=10, color='#1f77b4',
            arrowprops=dict(arrowstyle='->', color='#1f77b4'))

ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Throughput (M tokens/sec, log scale)', fontsize=11)

gpu_name = torch.cuda.get_device_name(0)
ax.set_title(
    f'moe-engine Router Throughput -- {gpu_name} (Triton) vs CPU Reference\n'
    f'(moe-engine v{pkg.__version__}, real measurements)',
    fontsize=12
)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

out_path = ARTIFACT_DIR / 'router_throughput_A100.png'
plt.tight_layout()
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Chart saved to {out_path}')

from IPython.display import Image
Image(str(out_path))


## 13. Production-scale sanity check (H=4096, E=64, K=2)

Matches `configs/default.yaml`'s scale directly (rather than a toy
dimension), on real A100 hardware.


In [ ]:
torch.manual_seed(42)
N, H, E, K = 2048, 4096, 64, 2

router = MoERouter(hidden_dim=H, num_experts=E, top_k=K).cuda()
tokens = torch.randn(N, H, device='cuda')

with torch.no_grad():
    idx, w, dispatch_cnt = router(tokens)

assert idx.shape == (N, K)
assert int(dispatch_cnt.sum()) == N * K
assert not torch.isnan(w).any()
assert (idx >= 0).all() and (idx < E).all()
assert torch.allclose(w.sum(-1), torch.ones(N, device='cuda'), atol=1e-4)

print('Production-scale Triton sanity check (default.yaml dimensions):')
print(f'  used_triton            : {router.last_profile.used_triton}')
print(f'  kernel_ms               : {router.last_profile.kernel_ms:.4f}')
print(f'  expert_load_imbalance   : {router.last_profile.expert_load_imbalance:.4f}')
print(f'  router_z_loss           : {router.last_profile.router_z_loss:.4f}')
print()
print(f'PASS: All invariants hold at H={H}, E={E}, K={K} on {torch.cuda.get_device_name(0)}.')


## 14. Expert capacity dropping at real hardware scale (E=256, K=8)

`configs/large_scale.yaml` (fine-grained MoE, 256 experts, top-8 routing)
was previously validated for correctness only at toy dimensions on CPU --
its behavior at real scale on real hardware was an explicitly open item
(see `roadmap.md` and the v3 preprint's Limitations section). This is the
first attempt to exercise it on real GPU hardware. If this cell fails
(e.g., out-of-memory), that is itself useful, honest information -- report
it as-is rather than reducing the config until it happens to pass.


In [ ]:
from pkg.utils.config import MoEConfig
from pkg.distributed.mesh import build_topology
from pkg.distributed.moe_layer import DistributedMoELayer

cfg = MoEConfig.from_yaml('configs/large_scale.yaml')
print(f"large_scale.yaml: E={cfg.model.num_experts}, K={cfg.model.top_k}, "
      f"H={cfg.model.hidden_dim}, capacity_dropping={cfg.model.capacity_dropping}, "
      f"capacity_factor={cfg.model.capacity_factor}")

topo = build_topology(dp_size=1, ep_size=1, device_type='cuda')

try:
    layer = DistributedMoELayer(
        hidden_dim=cfg.model.hidden_dim,
        ffn_dim=cfg.model.ffn_dim,
        num_experts=cfg.model.num_experts,
        top_k=cfg.model.top_k,
        topology=topo,
        capacity_factor=cfg.model.capacity_factor,
        capacity_dropping=cfg.model.capacity_dropping,
    ).cuda()

    B, S = 2, 512  # matches a modest slice of default.yaml-scale batching
    x = torch.randn(B, S, cfg.model.hidden_dim, device='cuda')

    torch.cuda.synchronize()
    t0 = torch.cuda.Event(enable_timing=True)
    t1 = torch.cuda.Event(enable_timing=True)
    t0.record()
    out = layer(x)
    t1.record()
    torch.cuda.synchronize()

    assert out.shape == x.shape
    assert not torch.isnan(out).any()

    peak_mem_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    print()
    print(f'RESULT: E=256, K=8 forward pass succeeded on real A100 hardware.')
    print(f'  Input shape            : {tuple(x.shape)}')
    print(f'  Forward latency        : {t0.elapsed_time(t1):.2f} ms')
    print(f'  Peak GPU memory        : {peak_mem_gb:.2f} GB')
    print(f'  dropped_token_fraction : {layer.last_dropped_token_fraction:.4f}')
    print(f'  overlap_ratio          : {layer.last_overlap_ratio:.4f}')

    out.sum().backward()
    print(f'  Backward pass           : succeeded, no NaN in gradients: '
          f'{not any(torch.isnan(p.grad).any() for p in layer.parameters() if p.grad is not None)}')

except torch.cuda.OutOfMemoryError as e:
    print(f'RESULT: E=256, K=8 hit an out-of-memory error at this batch size on this GPU.')
    print(f'  This is honest, useful information -- it means large_scale.yaml\'s')
    print(f'  real memory footprint at B={B}, S={S} exceeds what is available here.')
    print(f'  Error: {e}')
except Exception as e:
    print(f'RESULT: E=256, K=8 failed with a non-OOM error -- report this as a real finding:')
    print(f'  {type(e).__name__}: {e}')
    raise


## 15. Package all artifacts for download

In [ ]:
import shutil

zip_base = Path.cwd() / f'moe_engine_a100_validation_v{pkg.__version__}'
shutil.make_archive(str(zip_base), 'zip', ARTIFACT_DIR)
zip_path = f'{zip_base}.zip'
print(f'Artifacts zipped to: {zip_path}')
print()
print('Contents:')
for f in sorted(ARTIFACT_DIR.iterdir()):
    print(f'  {f.name}')


### Downloading from Lightning AI Studio

Lightning AI Studio doesn't have Colab's `files.download()` helper. Use
the Studio's file browser (left sidebar) to navigate to the zip file
printed above and download it directly, or use Lightning's own CLI/SDK if
you're scripting this outside the notebook UI.


---

## What to do with the downloaded zip

| File | Purpose |
|------|---------|
| `gpu_results_a100.json` | Raw A100 benchmark data |
| `cpu_results.json` | CPU reference path data for comparison |
| `router_throughput_A100.png` | Throughput chart — place in `benchmarks/charts/` |
| `test_suite_output.txt` | Full pytest output from a real-GPU environment |

If you found a new architecture-specific issue while running this
notebook, the right place for it is `docs/adr/ADR-005-gpu-architecture-portability.md`'s
support matrix — add a row, and if it needed a code change, document the
fix the same way Problems 1–3 in that ADR are documented, so the next
person on a new GPU generation inherits the lesson instead of
rediscovering it.
